In [ ]:
# data

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

mobility = pd.read_csv("data/mobility_clean.csv")
mobility['date'] = pd.to_datetime(mobility['date'])

In [ ]:
# overall cat trends

mobility_cats = mobility.copy()
mobility_cats["year_month"] = mobility_cats["date"].dt.to_period('M')


cat_cols = [
    "retail_and_recreation_percent_change_from_baseline", 
    "grocery_and_pharmacy_percent_change_from_baseline", 
    "transit_stations_percent_change_from_baseline", 
    "workplaces_percent_change_from_baseline", 
    "residential_percent_change_from_baseline"
]
monthly_mob = mobility_cats.groupby("year_month")[cat_cols].mean()
monthly_mob.index = monthly_mob.index.to_timestamp()

monthly_mob_long = (
    monthly_mob
    .reset_index()
    .melt(id_vars="year_month", value_vars=cat_cols,
          var_name="category", value_name="percent_change")
)

monthly_mob_long["category_short"] = monthly_mob_long["category"].replace({
    "retail_and_recreation_percent_change_from_baseline":"retail/recreation", 
    "grocery_and_pharmacy_percent_change_from_baseline":"grocery/pharmacy", 
    "transit_stations_percent_change_from_baseline":"transit", 
    "workplaces_percent_change_from_baseline":"workplaces", 
    "residential_percent_change_from_baseline":"residential"
    }
)

palette = sns.crayon_palette(["Cerulean", "Atomic Tangerine", "Brick Red", "Purple Mountains' Majesty", "Pine Green"])

fig, ax = plt.subplots(figsize=(12, 7))
sns.lineplot(
    data=monthly_mob_long, 
    x="year_month", 
    y="percent_change", 
    hue="category_short", palette=palette
)

sns.set_style(style="darkgrid")
ax.legend(title="Category", loc="upper right", fontsize=13, title_fontsize=15)


ax.set_title(" Monthly Percent Change from Baseline", fontsize=20)
ax.set_xlabel("Month", fontsize=15)
ax.set_ylabel("Percent Change (%)")
ax.grid(True) 
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.xticks(rotation=45, fontsize=15, style='italic')

plt.ylim(-50, 50)
plt.tight_layout() 
plt.show()
